In [ ]:
# cuedetat_myriad_student.py
#
# Trains a small CNN+MLP student on the (image, poke, cue_trajectory) Parquet
# generated by `cuedetat_myriad_distillation.py`. Exports ONNX FP16 sized for
# on-device deployment in the CueDetat Android app.
#
# Designed for Kaggle (T4 x2 or P100). Cells delimited with `# %%`.
#
# Inputs:  Kaggle dataset `hereliesaz/cuedetat-myriad-distillation`
# Outputs: `/kaggle/working/myriad_student_fp16.onnx` (target <2 MB)

# CueDetat MYRIAD student — training & ONNX export

Architecture:
  image  [3,128,128] -> CNN(4 layers) -> 128
  poke   [4]         -> MLP[128,64]   -> 64
  concat [192]       -> MLP[256,128]  -> [30,2]

Loss: weighted MSE, early steps weighted 2x.

## Cell 1 — Install + imports

In [ ]:
!pip install -q torch torchvision pandas pyarrow pillow tqdm \
    onnx onnxruntime onnxconverter-common matplotlib

In [ ]:
import os, glob, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Cell 2 — Locate Kaggle distillation dataset

In [ ]:
# Kaggle mounts attached datasets under /kaggle/input/<slug>/
DATA_ROOT = "/kaggle/input/cuedetat-trajectories"
PARQUET   = os.path.join(DATA_ROOT, "samples.parquet")
IMG_ROOT  = os.path.join(DATA_ROOT, "images")
assert os.path.exists(PARQUET), f"Missing {PARQUET}; attach the dataset."
df = pd.read_parquet(PARQUET)
print(f"Rows: {len(df)}  | first row keys: {list(df.columns)}")

## Cell 3 — Train / val split (90/10) + Dataset

In [ ]:
T_STEPS = 30
IMG_SIZE = 128

class TrajectoryDS(Dataset):
    def __init__(self, df, img_root):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_root, row["image_path"])).convert("RGB")
        arr = np.asarray(img, dtype=np.float32) / 127.5 - 1.0  # [-1, 1]
        img_t = torch.from_numpy(arr).permute(2, 0, 1).contiguous()  # [3, H, W]
        poke = torch.tensor(
            [row["poke_x"], row["poke_y"], row["poke_dx"], row["poke_dy"]],
            dtype=torch.float32,
        )
        traj = torch.tensor(row["trajectory"], dtype=torch.float32).view(T_STEPS, 2)
        return img_t, poke, traj

rng = np.random.default_rng(0)
perm = rng.permutation(len(df))
split = int(len(df) * 0.9)
train_df = df.iloc[perm[:split]]
val_df   = df.iloc[perm[split:]]
print(f"train={len(train_df)} val={len(val_df)}")

train_loader = DataLoader(
    TrajectoryDS(train_df, DATA_ROOT),
    batch_size=256, shuffle=True, num_workers=4, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    TrajectoryDS(val_df, DATA_ROOT),
    batch_size=256, shuffle=False, num_workers=4, pin_memory=True,
)

## Cell 4 — Student model

In [ ]:
class MyriadStudent(nn.Module):
    def __init__(self, t_steps=T_STEPS):
        super().__init__()
        self.t_steps = t_steps
        # CNN encoder: 128 -> 64 -> 32 -> 16 -> 8
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1, bias=False),
                nn.BatchNorm2d(o),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.cnn = nn.Sequential(
            block(3, 16),    # 64
            block(16, 32),   # 32
            block(32, 64),   # 16
            block(64, 128),  # 8
            nn.AdaptiveAvgPool2d(1),  # [B, 128, 1, 1]
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
        )
        self.poke_mlp = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.Linear(128 + 64, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, t_steps * 2),
        )

    def forward(self, img, poke):
        f_img  = self.cnn(img)            # [B, 128]
        f_poke = self.poke_mlp(poke)      # [B, 64]
        out    = self.head(torch.cat([f_img, f_poke], dim=-1))
        return out.view(-1, self.t_steps, 2)

model = MyriadStudent().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Student params: {n_params/1e6:.3f} M  ({n_params*4/1e6:.2f} MB at FP32)")

## Cell 5 — Train (80 epochs, AdamW, cosine LR, weighted MSE)

In [ ]:
EPOCHS = 80
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler() if DEVICE == "cuda" else None

# Weight earlier trajectory points more heavily
step_weights = torch.linspace(2.0, 0.5, T_STEPS).to(DEVICE).view(1, T_STEPS, 1)

def step(model, batch, train=True):
    img, poke, traj = [t.to(DEVICE, non_blocking=True) for t in batch]
    with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
        pred = model(img, poke)
        loss = ((pred - traj) ** 2 * step_weights).mean()
    if train:
        opt.zero_grad(set_to_none=True)
        if scaler:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()
    return loss.item()

history = []
best_val = float("inf")

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    train_losses = []
    for batch in train_loader:
        train_losses.append(step(model, batch, train=True))
    sched.step()

    model.eval()
    val_losses = []
    endpoint_errs = []
    with torch.no_grad():
        for batch in val_loader:
            val_losses.append(step(model, batch, train=False))
            img, poke, traj = [t.to(DEVICE, non_blocking=True) for t in batch]
            pred = model(img, poke)
            endpoint_errs.append(
                ((pred[:, -1] - traj[:, -1]) ** 2).sum(-1).sqrt().mean().item()
            )

    tl, vl = np.mean(train_losses), np.mean(val_losses)
    ee     = np.mean(endpoint_errs)
    history.append({"epoch": epoch, "train": tl, "val": vl, "endpoint_err": ee})
    print(f"epoch {epoch:3d} | train {tl:.5f} | val {vl:.5f} | "
          f"endpt_err {ee:.4f} | {time.time()-t0:.1f}s")

    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), "/kaggle/working/student_best.pt")

print("Best val loss:", best_val)

## Cell 6 — Visual eval: 16 random predictions vs ground truth

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/student_best.pt"))
model.eval()

import random
fig, axes = plt.subplots(2, 8, figsize=(24, 6))
val_ds = TrajectoryDS(val_df, DATA_ROOT)
sample_idxs = random.sample(range(len(val_ds)), 16)

with torch.no_grad():
    for ax, idx in zip(axes.flat, sample_idxs):
        img, poke, traj = val_ds[idx]
        pred = model(img.unsqueeze(0).to(DEVICE),
                     poke.unsqueeze(0).to(DEVICE))[0].cpu().numpy()
        ax.imshow((img.permute(1, 2, 0).numpy() + 1) / 2)
        gt = traj.numpy() * IMG_SIZE
        pr = pred * IMG_SIZE
        ax.plot(gt[:, 0], gt[:, 1], "g-", linewidth=2, label="teacher")
        ax.plot(pr[:, 0], pr[:, 1], "r--", linewidth=2, label="student")
        ax.plot(gt[0, 0], gt[0, 1], "go", markersize=6)
        ax.set_xlim(0, IMG_SIZE); ax.set_ylim(IMG_SIZE, 0)
        ax.axis("off")
axes[0, 0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("/kaggle/working/student_eval.png", dpi=100)
plt.show()

print(f"Final endpoint err on val: "
      f"{history[-1]['endpoint_err']:.4f} normalized table units "
      f"({history[-1]['endpoint_err']*100:.1f}% of table side)")
print("Acceptance: < 0.05 (5% of table diagonal). "
      f"{'PASS' if history[-1]['endpoint_err'] < 0.05 else 'FAIL'}")

## Cell 7 — Export ONNX FP16

In [ ]:
import onnx
from onnxconverter_common import float16

ONNX_FP32 = "/kaggle/working/myriad_student_fp32.onnx"
ONNX_FP16 = "/kaggle/working/myriad_student_fp16.onnx"

dummy_img  = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
dummy_poke = torch.randn(1, 4, device=DEVICE)

# Move to CPU for export to keep ORT mobile happy
model_cpu = model.cpu().eval()
torch.onnx.export(
    model_cpu,
    (dummy_img.cpu(), dummy_poke.cpu()),
    ONNX_FP32,
    input_names=["image", "poke"],
    output_names=["trajectory"],
    opset_version=17,
    dynamic_axes=None,  # static shapes for mobile
    do_constant_folding=True,
)
m_fp32 = onnx.load(ONNX_FP32)
m_fp16 = float16.convert_float_to_float16(m_fp32, keep_io_types=True)
onnx.save(m_fp16, ONNX_FP16)

size_fp32 = os.path.getsize(ONNX_FP32) / 1e6
size_fp16 = os.path.getsize(ONNX_FP16) / 1e6
print(f"ONNX FP32: {size_fp32:.2f} MB | ONNX FP16: {size_fp16:.2f} MB")
print(f"Acceptance: <2 MB. {'PASS' if size_fp16 < 2.0 else 'FAIL'}")

## Cell 8 — Verify ONNX inference matches PyTorch within tolerance

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(ONNX_FP16, providers=["CPUExecutionProvider"])
img_np  = dummy_img.cpu().numpy().astype(np.float32)
poke_np = dummy_poke.cpu().numpy().astype(np.float32)
ort_out = sess.run(["trajectory"], {"image": img_np, "poke": poke_np})[0]

with torch.no_grad():
    torch_out = model_cpu(dummy_img.cpu(), dummy_poke.cpu()).numpy()

max_diff = np.max(np.abs(ort_out - torch_out))
print(f"max abs diff (ONNX FP16 vs PyTorch FP32): {max_diff:.5f}")
print(f"Acceptance: < 0.01. {'PASS' if max_diff < 0.01 else 'FAIL — investigate quantization'}")
print("\nDownload /kaggle/working/myriad_student_fp16.onnx to ml/myriad_student_fp16.onnx")